# 05 - The photon transfer curve

**Purpose.** Run `protocols/02-ptc.md` and publish `g(gain)` - electrons per ADC count, per CFA
plane, at nine gain settings - together with the gain law fitted through it and the FPN test that
says whether sigma^2 is shot plus read and nothing else. It is the session that puts a scale on
every count session 01 measured.

**What it is not for.** Linearity, `ceiling(gain)` and full well: those need a characterised
light source, a shrunk ROI and a per-channel bend, and are a later session. Nor dark current.
Nothing here fits a bend, and no rung is placed to find one.

**Two halves, and they run at different times.** The first half scouts the bench and captures;
the second reads what is on disk and publishes to `results/`. They are separate cells on
purpose: re-running the analysis must never re-run the capture, because the frames cost a
bench night. `06_ptc_read` then explains what was published, and measures nothing itself.

## 1. Pre-flight, and the attenuation scout

**This section is `light-source.md` item 3, and it runs *warm*, before and apart from the session
proper.** It answers the one question the protocol cannot answer on paper: *what grey level and
how many sheets* put the bench where a twelve-rung ladder fits between the camera's shortest
exposure and a sane wall clock.

It is deliberately not gate 3. Gate 3 solves `t_sat(gain)` from the measured flux *inside* the
session, cold, with the bench undisturbed, and that is what the record quotes. This scout produces
a **bench configuration** - a sheet count and a grey level - plus a prediction of `t_sat` that
gate 3 re-measures. Nothing here is published to `results/`.

**Why it may run warm.** Flux is an optical measurement and the sensor's temperature does not
change how much light arrives. What temperature does change - dark current - is held down by
keeping the scout's exposures short, and none of its numbers survive into the record.

### The arithmetic that sets the target

Gain is in 0.1 dB units, so amplification is `10 ** (gain/200)` and **gain 450 is 178x gain 0**.
(600 would have been 1000x, and is out of this project's scope - `CLAUDE.md`, D52. Dropping it is
what brings the attenuation this bench needs from ~4000x down to ~800x.) With one fixed light
level `t_sat` spans 178:1 across the gain set, and the ladder spans another 300:1 (0.3% to 90%),
so the session's shortest exposure is

    0.003 * t_sat(450)  =  1.69e-5 * t_sat(0)

so `t_sat(0)` alone fixes the whole session: the margin over both floors below, and about
`25 * t_sat(0)` of shutter-open time.

**There are two floors, and the obvious one does not bind.** The camera's shutter stops at 32 us;
the panel's refresh period is 16.7 ms, **520x longer**. An exposure shorter than one period sees
whichever slice of the refresh cycle it lands in, and on a rolling-shutter sensor that slice
differs row by row - so the error arrives as spatial structure, which does *not* cancel in the
pair difference the way a uniform level drift does.

| `t_sat(0)` | faintest rung | margin over 32 us | rungs under one refresh period | session exposure |
|---|---|---|---|---|
| 1.9 s | 31 us | 1.0x | 55 of 96 | 0.6 min |
| 5 s | 81 us | 2.5x | 40 of 96 | 1.6 min |
| 12 s | 194 us | 6.1x | 27 of 96 | 3.9 min |
| 20 s | 323 us | 10.1x | 21 of 96 | 6.5 min |
| 30 s | 485 us | 15.2x | 17 of 96 | 9.7 min |

**Target: `t_sat(0)` between 10 and 30 s, aimed at the top of that window.** Wall clock is the
only thing the long end costs, and ~10 minutes of shutter-open against 464 frames of download and
save is not what makes this session long. Every other column improves.

**No configuration clears the refresh floor, which is why the flicker test below is a gate and
not a curiosity.** Emptying the last column would take `t_sat(0)` near 1000 s - seven hours of
shutter-open - so the high gains shoot sub-refresh rungs whatever we choose, and the question is
not how to avoid them but whether this panel is steady enough to make them harmless. An LCD holds
a static grey between refreshes and at 100% brightness its backlight is usually driven DC, in
which case exposure length is irrelevant and 0.5 ms is as good as 5 s. That is a claim about
*this* iPad, and the test measures it rather than assuming it.

**One level for all eight gains, not a level per gain.** The PTC plots variance against measured
signal, so re-attenuating between gains would be perfectly legitimate - but a fixed source makes
`t_sat(gain) * amplification(gain)` a constant, and that is a check on the 0.1 dB law from
timings alone, before a single variance is computed. Give it up only if the flicker test says the
high-gain rungs are unusable; the fallback is then to dim the panel for the top gains, paying
that check and a second flux in the record.


In [ ]:
import json
import pathlib
import sys
import time
import urllib.request

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, spatial, stats

RESULTS = pathlib.Path.cwd().parent / "results"

FULL_SCALE = 4095                    # ADC counts; the units rule in CLAUDE.md
GAINS = [0, 50, 100, 190, 200, 250, 300, 450]   # 450 is the ceiling (D52)
RUNGS = [0.3, 0.5, 0.85, 1.4, 2.4, 4.0, 6.8, 11.4, 19.2, 32, 54, 90]   # % of t_sat
ROI = (1408, 568, 1024, 1024)        # even origin and extent, or Bayer shifts (L05)
OFFSET = 15                          # project_offset, fixed by session 01
SCOUT_GAIN = 100
MAX_EXPOSURE = 2.0                   # s; the scout never needs a long frame

# The bench configuration this run measures.  A flux with no configuration
# beside it is not a measurement of anything (light-source.md item 3).
SHEETS = 8                           # sheets of paper between camera and panel
REFRESH_HZ = 60.0                    # panel refresh; one period is the flicker yardstick
PATCH_SERVER = "http://127.0.0.1:8765"

TARGET_TSAT0 = (10.0, 30.0)          # s, the window the table above argues for
AIM_TSAT0 = 30.0                     # aim at the long end: only wall clock pays for it

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
PEDESTAL_FIT = _bias["pedestal_fit"]["value"]
MIN_EXPOSURE = _bias["bias_exposure"]["value"]
HCG = _bias["hcg_threshold_gain"]["value"]


def amplification(gain):
    """Gain is in 0.1 dB units, so 200 units is exactly a factor of ten."""
    return 10.0 ** (gain / 200.0)


def pedestal(gain):
    """Published pedestal at offset 15, in ADC counts (03_bias_sweep)."""
    branch = PEDESTAL_FIT["hcg" if gain >= HCG else "lcg"]
    return branch["A"] + branch["B"] * amplification(gain)


def plane_means(mosaic):
    """Mean of each CFA plane, in ADC counts.  `to_adc` raises rather than
    truncate, so this doubles as a check that the frame came off the raw path."""
    return {k: float(stats.to_adc(v).mean()) for k, v in spatial.split(mosaic).items()}


def set_level(level, settle_s=0.8):
    """Drive `grey-patch.html` from here (protocols/patch-server.py).

    The page polls and applies the change, so the settle covers one poll plus a
    repaint.  Whether the panel actually followed is not taken on trust: the
    sweep below reads it back out of the pixels.
    """
    lvl = level if level == "free" else int(level)      # "free" hands the panel back
    with urllib.request.urlopen(f"{PATCH_SERVER}/set?level={lvl}", timeout=5) as r:
        state = json.load(r)
    time.sleep(settle_s)
    return state


print(f"bench: {SHEETS} sheets of paper, grey level driven from here")
print(f"min exposure {MIN_EXPOSURE * 1e6:.0f} us")
print("pedestal at offset 15:  " + "  ".join(f"g{g}={pedestal(g):.0f}" for g in GAINS))

### Gate 1 - white balance, verified from the pixels (L01)

The camera ships `WB_R=55`, `WB_B=75` and applies them to RAW16 before the data reaches us. The
control reading back as 50 proves only that the control took. The evidence is the modal step
between adjacent distinct values: **16 on all four planes**, greens at 16 with red 17/18 and blue
24 being the fingerprint of white balance still applied.

Nothing captured before this passes is usable - the scout included, because a smeared step means
`to_adc` refuses and every mean below is wrong.

In [ ]:
rig = asi.open_camera()
asi.neutralise_white_balance(rig)
asi.configure(rig, gain=SCOUT_GAIN, offset=OFFSET, roi=ROI)

dark, hdr = asi.capture(rig, MIN_EXPOSURE, imagetyp="DARK")
steps = {name: stats.value_step(p) for name, p in spatial.split(dark).items()}

print("modal value step per plane:", steps)
print("WB_R", rig.get("WB_R"), " WB_B", rig.get("WB_B"),
      " gain", rig.get("Gain"), " offset", rig.get("Offset"))
assert set(steps.values()) == {16}, f"gate 1 FAILED: {steps} -- stop, do not correct later"
print("\ngate 1 passed")

### Measuring a flux

One helper, used by everything below: step the exposure until the brightest plane lands near
mid-scale, then read flux off as `(mean - pedestal) / exptime`.

Mid-scale rather than near full scale on purpose. A plane close to 4095 is compressed by whatever
non-linearity lives near saturation, and this session is explicitly not the one that measures a
bend; half scale keeps the estimate on the part of the curve we are entitled to call straight.
One frame is discarded after every exposure change, as the protocol requires.

In [ ]:
ped_scout = pedestal(SCOUT_GAIN)


def auto_expose(start=1e-3, tries=9):
    """Land the brightest plane between 35% and 65% of full scale.

    Returns `(means, exptime, converged)`.  A run that does not converge is
    still returned rather than raised on: at the dim end the honest outcome is
    "as long as this scout will go and still not bright", and that is data.
    """
    exposure, means, exptime = start, None, start
    for _ in range(tries):
        asi.capture(rig, exposure, imagetyp="LIGHT")          # discard after the change
        mosaic, h = asi.capture(rig, exposure, imagetyp="LIGHT")
        means, exptime = plane_means(mosaic), h["EXPTIME"]
        top = max(means.values())
        if 0.35 * FULL_SCALE <= top <= 0.65 * FULL_SCALE:
            return means, exptime, True
        scale = 0.5 * FULL_SCALE / max(top - ped_scout, 1.0)
        nxt = min(max(exposure * scale, MIN_EXPOSURE), MAX_EXPOSURE)
        if abs(nxt - exposure) / exposure < 0.02:             # pinned at a limit
            break
        exposure = nxt
    return means, exptime, False


def flux_of(means, exptime):
    return {k: (v - ped_scout) / exptime for k, v in means.items()}


set_level(255)
means, exptime, ok = auto_expose()
flux = flux_of(means, exptime)
bright = max(flux, key=flux.get)

print(f"anchor: grey level 255, {SHEETS} sheets, gain {SCOUT_GAIN}, "
      f"{exptime * 1e6:.0f} us, converged={ok}")
for k, v in flux.items():
    print(f"  {k}: {v:12.1f} counts/s   ({v / flux[bright] * 100:5.1f}% of {bright})")

### What that flux implies

`t_sat(gain)` is where the **brightest** plane fills the headroom above its own pedestal -
brightest, because that is the plane that clips first and clipping is what the top rung must
avoid. The pedestal is not a detail here: at gain 600 it is over 1000 counts, a quarter of full
scale. At gain 450, the top of this project's range, it is 231 counts and the headroom is 3864.

In [ ]:
def tsat_table(flux_ref, label=""):
    amp_ref = amplification(SCOUT_GAIN)
    print(f"{label}\n{'gain':>5} {'amp':>8} {'pedestal':>9} {'headroom':>9} "
          f"{'t_sat':>12} {'0.3% rung':>12} {'90% rung':>12}")
    out = {}
    for g in GAINS:
        head = FULL_SCALE - pedestal(g)
        t = head / (flux_ref * amplification(g) / amp_ref)
        out[g] = t
        print(f"{g:5d} {amplification(g):8.1f} {pedestal(g):9.1f} {head:9.1f} "
              f"{t:11.4g}s {RUNGS[0] / 100 * t * 1e6:11.4g}u {RUNGS[-1] / 100 * t:11.4g}s")

    t0, ttop = out[0], out[GAINS[-1]]
    shortest = RUNGS[0] / 100 * ttop
    print(f"\nt_sat(gain 0)        {t0:12.4g} s    (target "
          f"{TARGET_TSAT0[0]:.0f}-{TARGET_TSAT0[1]:.0f} s)")
    print(f"shortest rung        {shortest * 1e6:12.4g} us   (floor {MIN_EXPOSURE * 1e6:.0f} us)")
    print(f"session shutter-open {25.1 * t0 / 60:12.4g} min")
    want = sum(TARGET_TSAT0) / 2
    print(f"attenuation to reach t_sat(0) = {want:.0f} s: {want / t0:.4g}x")
    return out


tsat_table(flux[bright], f"bench as it stands (level 255, {SHEETS} sheets; "
                         f"{bright} sets t_sat):")

### The grey-level curve, measured rather than assumed

L07 says grey level is exhausted below about 25% of full scale, because the backlight leaks
through a black LCD. That is a claim from a retired attempt about a different bench, and it is
cheap to check here: the level is driven from this notebook, so the curve costs no trips to the
iPad.

Two things come out of it. The **usable range** - how much attenuation the level alone can
deliver before the leak floor - and a check that the panel is actually following: if the page is
not connected to the server, every level returns the same flux and the table below is flat.

In [ ]:
LEVELS = [255, 224, 192, 160, 128, 96, 64, 48, 32, 24, 16, 8, 0]

curve = []
for lv in LEVELS:
    set_level(lv)
    m, e, converged = auto_expose(start=max(exptime, MIN_EXPOSURE))
    curve.append((lv, flux_of(m, e)[bright], e, converged))

f255 = curve[0][1]
print(f"{'level':>6} {'% of white':>11} {'flux':>14} {'attenuation':>12} "
      f"{'exptime':>10} {'converged':>10}")
for lv, f, e, converged in curve:
    print(f"{lv:6d} {lv / 255 * 100:10.1f}% {f:14.1f} {f255 / max(f, 1e-9):11.4g}x "
          f"{e * 1e6:9.0f}u {str(converged):>10}")

best = curve[-1][1]
print(f"\ngrey level alone gives {f255 / max(best, 1e-9):.4g}x, from 255 down to 0.")
assert curve[-1][1] < 0.5 * f255, ("level 0 is as bright as level 255 -- the panel is not "
                                   "following the server.  Reload grey-patch.html on the iPad.")

### Choosing the configuration

The curve above is a set of measured fluxes; this is the one decision the scout exists to make.
Each level implies a `t_sat(0)`, and each `t_sat(0)` implies all 96 exposures the session would
shoot - so the choice is judged against the whole ladder rather than against a single number.

No interpolation. The level sweep steps by roughly 1.6x in flux, the target window is 3x wide,
and a level that was measured is worth more than one that was fitted. If nothing lands inside the
window the sheet stack is the wrong thickness: change it, re-run the scout, and **never** scale
the flux by a per-sheet figure (L07, L08 - stacked diffusers give diminishing returns).

The ladder it prints is capped at the top by `top_rung`: a rung whose own noise no longer fits
under 4095 measures a variance that is too low and a `g` that is too high, and the pair difference
cannot see through it. That is a protocol rule (`02-ptc.md`), found by this scout on 2026-08-31 -
gain 450 read `g = 0.0593` at 90% against 0.0511 at its clean rungs - and it moves two rungs: 90%
becomes 89% at gain 300 and **75% at gain 450**.


In [ ]:
period = 1.0 / REFRESH_HZ
N_RUNGS = len(GAINS) * len(RUNGS)
CLIP_SIGMAS = 4.0                     # of a frame's own noise, under the top code


def g_predicted(gain):
    """L25's gain anchor carried by L29's 0.1 dB slope, in e- per ADC count.

    A prediction, used to place a rung and nothing else.  It never enters a
    published number, and it errs safe: a larger true `g` only widens the margin
    the cap below is protecting.
    """
    return 9.382 * 10 ** (-0.00502 * gain)


def top_rung(gain):
    """The highest rung, in % of headroom, whose own noise still fits under 4095.

    Solve `S + k*sqrt(S/g) = headroom` for S.  Above it the bright tail of the
    frame's own spread is censored by the top code, the measured variance comes
    in low, and that reads as a `g` that is too high -- which pair-differencing
    cannot see through, because the clipping happens before the subtraction
    (protocols/02-ptc.md).  It binds only where `g` is small: 90% everywhere up
    to gain 250, 89% at 300, 75% at 450.
    """
    head = FULL_SCALE - pedestal(gain)
    gg = g_predicted(gain)
    s = (-CLIP_SIGMAS / np.sqrt(gg) + np.sqrt(CLIP_SIGMAS ** 2 / gg + 4 * head)) / 2
    return min(RUNGS[-1], 100 * s * s / head)


def ladder_for(t0):
    """Every exposure the session shoots, in seconds, keyed by gain.

    One light level and eight gains leaves no free parameter: `t_sat` scales as
    the headroom above that gain's pedestal, divided by the amplification.  The
    top rungs are capped by `top_rung`, so the ladder is 0.3% to 90% of `t_sat`
    only where 90% is a rung that can be measured.
    """
    head0 = FULL_SCALE - pedestal(0)
    out = {}
    for g in GAINS:
        t_sat = t0 * (FULL_SCALE - pedestal(g)) / head0 / amplification(g)
        cap = top_rung(g)
        out[g] = [min(r, cap) / 100 * t_sat for r in RUNGS]
    return out


def tsat0_of(flux_scout):
    """t_sat at gain 0 implied by a flux measured at SCOUT_GAIN."""
    return (FULL_SCALE - pedestal(0)) / (flux_scout / amplification(SCOUT_GAIN))


print(f"{'level':>6} {'t_sat(0)':>10} {'faintest':>11} {'x floor':>9} "
      f"{'sub-refresh':>13} {'in window':>10}")
choices = []
for lv, f, e, converged in curve:
    if f <= 0:
        continue
    t0 = tsat0_of(f)
    rungs = ladder_for(t0)
    shortest = min(min(v) for v in rungs.values())
    n_sub = sum(1 for v in rungs.values() for x in v if x < period)
    inside = TARGET_TSAT0[0] <= t0 <= TARGET_TSAT0[1]
    choices.append((lv, t0, inside))
    print(f"{lv:6d} {t0:9.4g}s {shortest * 1e6:10.4g}u {shortest / MIN_EXPOSURE:8.1f}x "
          f"{n_sub:8d}/{N_RUNGS:<4d} {str(inside):>10}")

usable = [c for c in choices if c[2]]
if usable:
    level, t_sat0, _ = min(usable, key=lambda c: abs(c[1] - AIM_TSAT0))
    print(f"\nbench configuration: grey level {level}, {SHEETS} sheets, "
          f"t_sat(0) = {t_sat0:.4g} s")
else:
    level, t_sat0, _ = min(choices, key=lambda c: abs(np.log(c[1] / AIM_TSAT0)))
    print(f"\nNo level lands in {TARGET_TSAT0[0]:.0f}-{TARGET_TSAT0[1]:.0f} s; closest is "
          f"level {level} at {t_sat0:.4g} s.")
    print(f"The stack is off by {AIM_TSAT0 / t_sat0:.3g}x in attenuation -- "
          f"{'add' if t_sat0 < AIM_TSAT0 else 'remove'} sheets, re-run the scout, and do not "
          f"predict the new flux from a per-sheet figure.")

flux_at = {lv: f for lv, f, _, _ in curve}
brighter = [lv for lv, _, _ in choices if lv > level]
if brighter and flux_at[min(brighter)] / flux_at[level] < 1.2:
    print(f"NOTE: level {level} is on the leak floor -- level {min(brighter)} is only "
          f"{flux_at[min(brighter)] / flux_at[level]:.2f}x brighter, so the level is not "
          f"driving the flux and there is no trim in this direction.  It is the backlight "
          f"leaking through a black LCD (L07), not a level anyone set.  The configuration "
          f"works; adding sheets is what moves it back onto the responsive part of the curve.")

lad = ladder_for(t_sat0)
head0 = FULL_SCALE - pedestal(0)
print(f"\n{'gain':>5} {'t_sat':>10} {'faintest rung':>15} {'top rung':>11} {'of t_sat':>9} "
      f"{'sub-refresh':>12} {'x shutter floor':>16}")
for g in GAINS:
    v = lad[g]
    t_sat = t_sat0 * (FULL_SCALE - pedestal(g)) / head0 / amplification(g)
    print(f"{g:5d} {t_sat:9.4g}s {v[0] * 1e3:14.4g}ms {v[-1]:10.4g}s {top_rung(g):8.0f}% "
          f"{sum(1 for x in v if x < period):9d}/{len(RUNGS):<2d} {v[0] / MIN_EXPOSURE:15.1f}x")

print(f"\nblock 1 shutter-open {sum(4 * sum(v) for v in lad.values()) / 60:.1f} min; "
      f"block 2's bias frames are {MIN_EXPOSURE * 1e6:.0f} us each and cost nothing.")
print(f"faintest rung of the session: {min(min(v) for v in lad.values()) * 1e6:.0f} us at gain "
      f"{GAINS[-1]}, {min(min(v) for v in lad.values()) / period:.3f} of a refresh period.")


### Does the panel flicker? - the gate on the high-gain rungs

The table above says how many rungs are shorter than one refresh period; this says whether that
matters. The test is run **at the gain and the exposures the session will actually use**, because
flicker is not a property of the panel alone but of the panel, the exposure length and a rolling
shutter together.

It needs no knowledge of `g`, and it is not a mean-jitter test: a uniform level shift between two
frames cancels in a pair difference taken about its own mean, so the thing that would survive
into the PTC is *structure* - a slice of the refresh cycle that differs row by row. Two numbers
per rung, then:

- **`g_est = S / (var - R^2)`**, with `R` in counts from session 01's bias sweep. If the panel is
  steady this is the same number at every exposure - it is `g` - and if short exposures carry an
  extra variance it sags at the short end.
- **the row ratio**: the scatter of row means of the pair difference over what that scatter would
  be if the pixels were independent. One means no row structure; above one means something
  scanned while the shutter was open.

A sag at the short end refutes "one level for all eight gains" and sends the session to a per-gain
level (see the argument above). A flat table licenses the whole ladder.

**A rung within 4 sigma of the ceiling is excluded from that verdict.** At gain 450 one frame's
own noise is hundreds of counts, so a bright rung clips its own shot-noise tail, the pair variance
comes in low, and `g_est` reads high - a variance deficit that has nothing to do with the panel.
The headroom column says how many sigmas of margin each rung has.


In [ ]:
FLICKER_GAIN = GAINS[-1]
PROBE_RUNGS = [0, 2, 4, 6, 9, 11]     # faintest to top, spanning sub- to multi-refresh


def read_noise_counts(gain, offset=OFFSET):
    """Session 01's measured R at the nearest swept gain, in ADC counts."""
    rows = np.genfromtxt(RESULTS / "bias_sweep.csv", delimiter=",", names=True)
    rows = rows[rows["offset"] == offset]
    return float(rows["R_at_offset"][np.argmin(abs(rows["gain"] - gain))])


def pair_stats(exposure_s, n_pairs=3):
    """Signal, pair-difference variance and row structure at one exposure.

    Rule 2 of the protocol's analysis rules, run early and on one plane: the
    variance of a frame pair's difference, halved.  Two diagnostics come free
    from the same frames.  The row ratio is what a scanning panel or a backlight
    switching mid-readout shows up in, because neither is uniform over rows.
    The mean CV is the loudest PWM canary: a backlight that is off for part of
    a sub-refresh exposure moves the whole frame, and while a uniform move
    largely cancels in the pair difference it is the thing to see before
    trusting a dimmed panel.
    """
    asi.capture(rig, exposure_s, imagetyp="LIGHT")            # discard after the change
    sig, var, rows, frames = [], [], [], []
    for _ in range(n_pairs):
        a, _ = asi.capture(rig, exposure_s, imagetyp="LIGHT")
        b, _ = asi.capture(rig, exposure_s, imagetyp="LIGHT")
        pa = stats.to_adc(spatial.split(a)[bright]).astype(float)
        pb = stats.to_adc(spatial.split(b)[bright]).astype(float)
        d = pa - pb
        frames += [pa.mean(), pb.mean()]
        sig.append(0.5 * (pa.mean() + pb.mean()) - pedestal(FLICKER_GAIN))
        var.append(d.var(ddof=1) / 2.0)
        rows.append(d.mean(axis=1).std(ddof=1) / (d.std(ddof=1) / np.sqrt(d.shape[1])))
    frames = np.array(frames)
    return np.mean(sig), np.mean(var), np.mean(rows), frames.std(ddof=1) / frames.mean() * 100


set_level(level)
asi.configure(rig, gain=FLICKER_GAIN, offset=OFFSET, roi=ROI)
asi.capture(rig, lad[FLICKER_GAIN][0], imagetyp="LIGHT")      # 2 discards after a gain change
asi.capture(rig, lad[FLICKER_GAIN][0], imagetyp="LIGHT")
R = read_noise_counts(FLICKER_GAIN)

print(f"gain {FLICKER_GAIN}, grey level {level}, R = {R:.3f} counts, "
      f"one refresh period = {period * 1e3:.1f} ms\n")
head450 = FULL_SCALE - pedestal(FLICKER_GAIN)
print(f"{'exposure':>11} {'periods':>9} {'signal':>9} {'pair var':>10} "
      f"{'g_est':>9} {'row ratio':>10} {'mean CV':>9} {'headroom':>9}")
probe = []
for i in PROBE_RUNGS:
    e = lad[FLICKER_GAIN][i]
    if e < MIN_EXPOSURE:
        print(f"{e * 1e6:9.0f} us  below the {MIN_EXPOSURE * 1e6:.0f} us shutter floor -- skipped")
        continue
    S, var, row, cv = pair_stats(e)
    g_est = S / max(var - R ** 2, 1e-9)
    margin = (head450 - S) / np.sqrt(var)          # in sigmas of one frame's own noise
    probe.append((e, g_est, row, cv, margin))
    print(f"{e * 1e3:9.3f} ms {e / period:9.3f} {S:9.1f} {var:10.1f} "
          f"{g_est:9.4f} {row:10.3f} {cv:8.3f}% {margin:8.1f}s"
          f"{'  <- tail clipping, excluded' if margin < 4 else ''}")

clean = [r for r in probe if r[4] >= 4]          # a clipped tail is a variance deficit,
short = [g for e, g, _, _, _ in clean if e < period]      # not a flicker signal
long_ = [g for e, g, _, _, _ in clean if e >= period]
if short and long_:
    sag = np.mean(short) / np.mean(long_)
    print(f"\nsub-refresh g_est / multi-refresh g_est = {sag:.3f}")
    print("flat to a few percent: the ladder is licensed as it stands." if abs(sag - 1) < 0.05
          else "SAGGING: the short rungs carry a variance the long ones do not.  One level for "
               "all eight gains is refuted -- dim the panel for the top gains and record why.")
print(f"worst row ratio {max(r for _, _, r, _, _ in probe):.3f} "
      f"(1.0 is independent pixels; a scanning panel reads high)")
cv_short = max((c for e, _, _, c, _ in probe if e < period), default=0.0)
cv_long = max((c for e, _, _, c, _ in probe if e >= period), default=0.0)
print(f"worst frame-to-frame mean CV: {cv_short:.3f}% sub-refresh, {cv_long:.3f}% above.")
print("A dimmed backlight is the case to watch: PWM puts percent-level scatter into the short"
      "\nexposures and none into the long ones, and it is the reason brightness may not be"
      "\nlowered without re-running this cell.")


### Record for the bench

The scout ends with a **configuration, not a constant**: the grey level chosen above, the
sheet count, and the flux that pair produced, written into the session record with the ambient temperature. The attenuation
is valid only while nobody moves the camera off the panel.

If the attenuation the bench can reach falls short of what `t_sat(0)` needs, the honest response
is not to fudge the ladder. It is to say which gains the source can support and shorten the gain
set at the top - and to record that gain 600 was dropped for want of light rather than quietly
shooting it with a ladder whose bottom rungs sit under the shutter floor.

In [ ]:
set_level(level)
rig.close()      # drops the cooler too, but the scout never turned it on
print("camera closed")

In [ ]:
import datetime as dt

DATA = pathlib.Path.cwd().parent / "data" / "session02"
DATA.mkdir(parents=True, exist_ok=True)
BENCH_FILE = DATA / "bench.json"

# The three fields no instrument can read back.  Fill them in from the room and
# the iPad before running this cell -- brightness in particular, because the
# slider has no readback and the flux below is only reproducible with it.
BENCH = {
    "ambient_C": "25",
    "brightness_pct": 30,
    "panel_warmup_started": "00:10",        # "HH:MM", ten minutes before the first frame
}

BENCH.update({
    "grey_level": int(level),
    "sheets": SHEETS,
    "scout_gain": SCOUT_GAIN,
    "bright_plane": bright,
    "flux_counts_per_s": {str(lv): f for lv, f, _, _ in curve},
    "t_sat0_predicted_s": float(t_sat0),
    "t_sat_predicted_s": {str(g): v[-1] / (min(RUNGS[-1], top_rung(g)) / 100)
                          for g, v in lad.items()},
    "roi": list(ROI),
    "offset": OFFSET,
    "measured_on": dt.date.today().isoformat(),
})

assert all(BENCH[k] is not None for k in ("ambient_C", "brightness_pct")), \
    "fill in the room and the iPad first -- a flux with no brightness beside it is not reproducible"

BENCH_FILE.write_text(json.dumps(BENCH, indent=2) + "\n")
print(f"bench configuration written to {BENCH_FILE}")
print(f"  grey level {BENCH['grey_level']}, {BENCH['sheets']} sheets, "
      f"brightness {BENCH['brightness_pct']}%, t_sat(0) = {BENCH['t_sat0_predicted_s']:.4g} s")
print("\nThis is data/, not results/: it is the session's own record of a configuration,")
print("not a published constant.  Gate 3 re-measures every number in it, cold.")

## 2. The session

Everything from here runs **cold**, in this same kernel, against `protocols/02-ptc.md`. The scout
above closed the camera warm; this opens it again, cools it, and re-measures from the sensor what
the scout predicted from a warm one.

**A dead run is restarted, not resumed** - the same rule session 01 settled on. Delete
`data/session02/frames/` and go again: the cooler has to settle from scratch anyway, and
resumption logic is code that runs once, in the dark, under time pressure, having never been
tested on the case it exists for. `fits.write` refuses to overwrite, so a restart into a
directory that still holds frames stops immediately rather than blending two runs.

**If the kernel is fresh**, run cells 2 and 6 of the scout first - they define the constants,
`pedestal`, `plane_means` and `set_level` that everything below uses - then this section reads
the bench configuration back from `data/session02/bench.json` rather than from kernel state.

In [ ]:
import pandas as pd

from astropix import fits as F

FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

SETPOINT_C = asi.SETPOINT_C     # -10 C, and not a free parameter here
DISCARD = 2                     # frames dropped after a gain change (protocol)
DISCARD_EXPOSURE = 1            # after an exposure change
N_LADDER = 4                    # per rung: two disjoint pairs, so the variance has a repeat
N_BIAS = 10                     # per gain, block 2
FRAME_GAP_S = 0.2               # readout, USB and the file write are what heat the sensor
HOLD_TIMEOUT_S = 300.0          # a hold that never ends is a cooler fault, not a wait
MAX_RETAKES = 10                # per frame slot; more than this is not a transient
BIAS_LEVEL = 0                  # block 2 shoots with the panel driven black

bench = json.loads(BENCH_FILE.read_text())
LEVEL = bench["grey_level"]

planned = len(GAINS) * (len(RUNGS) * N_LADDER + N_BIAS)
print(f"bench: grey level {LEVEL}, {bench['sheets']} sheets, "
      f"brightness {bench['brightness_pct']}%, recorded {bench['measured_on']}")
print(f"{planned} frames planned, {planned * ROI[2] * ROI[3] * 2 / 1e9:.2f} GB on C:")
print(f"{len(list(FRAMES.glob('*.fits')))} frames already on disk")

### Opening the camera, and gate 1 again

The scout passed gate 1 warm, in a different camera session. This is a new open, and the rule is
that nothing captured before the check passes is usable - so it is checked again rather than
remembered. It costs five 32 us frames.

Three things are read off the device rather than assumed and all three go in the session record:
the control ranges, the minimum exposure, and the white balance the camera shipped with.

In [ ]:
existing = list(FRAMES.glob("*.fits"))
assert not existing, (f"{len(existing)} frames already in {FRAMES} -- delete the directory "
                      "to restart, or skip to the analysis half")

rig = asi.open_camera()
print("gain range     ", rig.range("Gain"))
print("offset range   ", rig.range("Offset"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
asi.configure(rig, gain=SCOUT_GAIN, offset=OFFSET)
BIAS_EXPOSURE = rig.min_exposure_s()          # measured, never assumed

for _ in range(DISCARD):
    asi.capture(rig, BIAS_EXPOSURE)

gate1 = {}
for _ in range(5):
    mosaic, _ = asi.capture(rig, BIAS_EXPOSURE)
    for name, plane in spatial.split(mosaic).items():     # stored values: the grid is 16 there
        gate1.setdefault(name, []).append(stats.value_step(plane))

for name in spatial.PLANES:
    print(f"  {name:2s} modal step {gate1[name]}")
bad = {n: s for n, s in gate1.items() if set(s) != {16}}
assert not bad, (f"white balance is still being applied: {bad} -- stop the session, "
                 "nothing captured from here is usable (L01)")
print(f"\ngate 1 passed on five frames.  bias exposure {BIAS_EXPOSURE * 1e6:.0f} us")

### Gate 2 - the cooler holds

-10 C, held in band for a continuous **30 seconds** before the first frame, judged by the
temperature trend and not by duty cycle. Measured 2026-08-28: this TEC approaches monotonically
and does not ring, which is why thirty seconds is enough.

Every reading is written to disk as it is taken. The sensor only reports while we are the ones
cooling it, so a reading not persisted is a reading gone.

**Cooling changes the light nothing and the frames everything.** The panel is on the other side
of the glass; what the cooler buys is a dark current small enough to ignore over a 21 s rung and
a read noise that matches the `R(gain)` session 01 published at this setpoint - which is the
number the analysis passes in rather than fitting (L10).

In [ ]:
cool_log = open(DATA / "cooldown.csv", "w", newline="")
cool_log.write("elapsed_s,temp_C,duty_pct\n")


def show(elapsed, temp, duty):
    cool_log.write(f"{elapsed},{temp},{duty}\n")
    cool_log.flush()                     # the reading exists nowhere else
    if int(elapsed) % 30 == 0:
        print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)


try:
    trace = asi.cool_to(rig, SETPOINT_C, log=show)
finally:
    cool_log.close()

temps = [t for _, t, _ in trace if t is not None]
print(f"\nsettled in {trace[-1][0]:.0f} s;  {temps[0]} C -> {temps[-1]} C, "
      f"minimum {min(temps)} C")
print(f"duty at setpoint {trace[-1][2]}%  <- the headroom this room leaves")

### Gate 3 - `t_sat(gain)`, measured cold at every gain

The scout predicted `t_sat` by scaling one warm flux with the 0.1 dB law. This measures it at
each of the eight gains instead, which is what the protocol asks for: *never extrapolate the
saturating exposure*. The two numbers being close is a result and not an assumption - it is a
check on the gain law from timings alone, before any variance is computed, and it is only
available because one grey level serves all eight gains.

The probe lands the brightest plane between 20% and 70% of its headroom. Below full scale on
purpose: this session is not the one that measures a bend, so no probe is placed where a bend
would live.

The pedestal used here is session 01's fit, which is right for choosing an exposure and is *not*
what the analysis will use - that comes from this session's own block 2, shot minutes away from
each ladder (L14).

In [ ]:
def measure_flux(gain, guess_s, tries=5):
    """Counts per second on the brightest plane, at one gain, cold.

    Returns `(flux, plane, exptime, converged)`.  One frame is discarded after
    every exposure change, as the protocol requires.
    """
    asi.configure(rig, gain=gain, offset=OFFSET)
    for _ in range(DISCARD):
        asi.capture(rig, guess_s, imagetyp="FLAT")

    head, e = FULL_SCALE - pedestal(gain), guess_s
    for _ in range(tries):
        asi.capture(rig, e, imagetyp="FLAT")               # discard after the change
        mosaic, h = asi.capture(rig, e, imagetyp="FLAT")
        means = plane_means(mosaic)
        plane = max(means, key=means.get)
        signal = means[plane] - pedestal(gain)
        if 0.2 * head <= signal <= 0.7 * head:
            return signal / h["EXPTIME"], plane, h["EXPTIME"], True
        e = min(max(e * 0.45 * head / max(signal, 1.0), MIN_EXPOSURE), 60.0)
    return signal / h["EXPTIME"], plane, h["EXPTIME"], False


def ladder_from_tsat(t_sat):
    """The rungs, in seconds, from a measured `t_sat` per gain.

    Same rungs and the same `top_rung` cap as the scout used -- only the scale
    is different, because this one was measured rather than predicted.
    """
    return {g: [min(r, top_rung(g)) / 100 * t_sat[g] for r in RUNGS] for g in t_sat}


set_level(LEVEL)
t_sat, flux, gate3 = {}, {}, []
predicted = {int(g): v for g, v in bench["t_sat_predicted_s"].items()}

print(f"{'gain':>5} {'plane':>6} {'probe':>10} {'flux':>12} {'t_sat':>10} "
      f"{'scout said':>11} {'ratio':>7}")
for g in GAINS:
    f_g, plane, e, ok = measure_flux(g, max(0.4 * predicted[g], MIN_EXPOSURE))
    flux[g], t_sat[g] = f_g, (FULL_SCALE - pedestal(g)) / f_g
    gate3.append({"gain": g, "plane": plane, "probe_s": e, "flux": f_g,
                  "t_sat_s": t_sat[g], "converged": ok})
    print(f"{g:5d} {plane:>6} {e * 1e3:9.3f}ms {f_g:12.1f} {t_sat[g]:9.4g}s "
          f"{predicted[g]:10.4g}s {t_sat[g] / predicted[g]:7.3f}"
          f"{'' if ok else '   <- probe did not converge'}")

lad_cold = ladder_from_tsat(t_sat)
faintest = min(min(v) for v in lad_cold.values())
assert faintest >= MIN_EXPOSURE, (f"faintest rung {faintest * 1e6:.0f} us is under the "
                                  f"{MIN_EXPOSURE * 1e6:.0f} us shutter floor -- the bench is "
                                  "too bright, and no ladder fixes that")

shutter = sum((N_LADDER + DISCARD_EXPOSURE) * sum(v) for v in lad_cold.values())
print(f"\nfaintest rung {faintest * 1e6:.0f} us, {faintest / MIN_EXPOSURE:.1f}x the floor")
readouts = planned + len(GAINS) * (DISCARD + len(RUNGS) + DISCARD_EXPOSURE)
print(f"shutter-open {shutter / 60:.1f} min over {readouts} readouts at "
      f"~{0.107 + FRAME_GAP_S:.1f} s each: {(shutter + readouts * 0.307) / 60:.0f} min in all")
print("gain-law check from timings alone: the ratio column should be flat, and any tilt in it")
print("is the 0.1 dB law disagreeing with this camera before a single variance is computed.")
pd.DataFrame(gate3).to_csv(DATA / "gate3.csv", index=False)
print(f"\ngate 3 written to {DATA / 'gate3.csv'} (data/, not results/)")

### The capture loop

Three functions, and the notebook's only loop over frames - the library does one frame, this does
the session.

**Temperature is enforced by retaking.** A frame whose own header says it was shot outside the
band is not written; it is thrown away and taken again. Unlike session 01's 32 us bias frames a
retake here costs a real exposure, so the budget binds sooner: **10 retakes of one frame slot, or
a hold past 300 s, stops the session** and names the cause.

**A warm reading holds the run.** Warm means the readout is heating the sensor faster than the
cooler removes it, which does not fix itself, so the run shoots discards at the same exposure
until the temperature has been back in band for a continuous `asi.RECOVER_S`. The discards are
the point: idling would let the cooler wind back to its idle duty, and resuming would re-run the
transient being recovered from. The cold side is left to pass on its own - that is the TEC
undershooting.

**Discards follow the protocol exactly**: two after every gain change, one after every exposure
change, and none of them written.

In [ ]:
def hold_for_temperature(where, exposure_s):
    """Shoot discards until the sensor has been in band for `asi.RECOVER_S`.

    Returns the seconds lost.  Discards are shot at the run's own exposure, so
    the duty cycle during the hold is the duty cycle being recovered from.
    """
    t0, in_band_since, worst = time.monotonic(), None, None
    while True:
        _, header = asi.capture(rig, exposure_s, imagetyp="FLAT")     # discarded on purpose
        time.sleep(FRAME_GAP_S)
        temp, now = header["CCD-TEMP"], time.monotonic()
        worst = temp if worst is None or (temp is not None and temp > worst) else worst

        if temp is not None and abs(temp - SETPOINT_C) <= asi.BAND_C:
            in_band_since = now if in_band_since is None else in_band_since
            if now - in_band_since >= asi.RECOVER_S:
                held = now - t0
                print(f"      held {held:.0f} s at {where}, peak {worst} C, "
                      f"back at {temp} C", flush=True)
                return held
        else:
            in_band_since = None

        if now - t0 > HOLD_TIMEOUT_S:
            raise TimeoutError(
                f"{HOLD_TIMEOUT_S:.0f} s of holding at {where} and still {temp} C: the "
                "cooler is not keeping up.  Stop the session and check the ambient and "
                "the fan -- do not widen the band")


def capture_frames(n, exposure_s, imagetyp, name, where):
    """`n` in-band frames at one setting, written as `name`_000.fits onwards.

    Returns `(retaken, held_s)`.  The exposure change is already paid for by the
    caller's discard, so this writes exactly `n` frames that were in band when
    the camera itself reported the temperature.
    """
    retaken, held_s = 0, 0.0
    for i in range(n):
        for _ in range(MAX_RETAKES + 1):
            mosaic, header = asi.capture(rig, exposure_s, imagetyp=imagetyp)
            temp = header["CCD-TEMP"]
            if temp is not None and abs(temp - SETPOINT_C) <= asi.BAND_C:
                break
            retaken += 1
            print(f"    ! {where} frame {i} at {temp} C, retaking", flush=True)
            if temp is not None and temp > SETPOINT_C + asi.BAND_C:
                held_s += hold_for_temperature(where, exposure_s)
            else:
                time.sleep(FRAME_GAP_S)
        else:
            raise RuntimeError(
                f"{MAX_RETAKES} retakes at {where} and still {temp} C.  This is no longer "
                "a transient -- stop the session rather than filling the curve with frames "
                "nobody can defend")

        F.write(FRAMES / f"{name}_{i:03d}.fits", mosaic, header)
        time.sleep(FRAME_GAP_S)
    return retaken, held_s


def run_gain(gain):
    """One gain: its twelve rungs, then its own pedestal block, shot adjacent.

    Block 2 is not a re-measurement of read noise -- it is the pedestal this
    gain's signal is measured against, taken minutes rather than hours away
    (L14).  It shoots with the panel driven black, and what still gets through
    is reported rather than assumed.
    """
    t0, retaken, held = time.monotonic(), 0, 0.0
    asi.configure(rig, gain=gain, offset=OFFSET)
    for _ in range(DISCARD):
        asi.capture(rig, lad_cold[gain][0], imagetyp="FLAT")

    for ri, exposure_s in enumerate(lad_cold[gain]):
        for _ in range(DISCARD_EXPOSURE):
            asi.capture(rig, exposure_s, imagetyp="FLAT")
        r, h = capture_frames(N_LADDER, exposure_s, "FLAT",
                              f"flat_g{gain:03d}_r{ri:02d}",
                              f"gain {gain} rung {ri} ({exposure_s * 1e3:.1f} ms)")
        retaken, held = retaken + r, held + h
        print(f"    rung {ri:>2}/{len(RUNGS)}  {exposure_s * 1e3:9.3f} ms  "
              f"{(time.monotonic() - t0) / 60:5.1f} min", flush=True)

    set_level(BIAS_LEVEL)
    for _ in range(DISCARD_EXPOSURE):
        asi.capture(rig, BIAS_EXPOSURE, imagetyp="BIAS")
    r, h = capture_frames(N_BIAS, BIAS_EXPOSURE, "BIAS", f"bias_g{gain:03d}",
                          f"gain {gain} pedestal")
    retaken, held = retaken + r, held + h
    set_level(LEVEL)

    print(f"  gain {gain}: {len(RUNGS) * N_LADDER + N_BIAS} frames, {retaken} retaken, "
          f"{held / 60:.1f} min held, {(time.monotonic() - t0) / 60:.1f} min\n", flush=True)
    return retaken, held

### Blocks 1 and 2 - the ladder, and the pedestal beside it

| block | gains | rungs | frames each |
|---|---|---|---|
| 1 - ladder | the eight | 12, geometric, capped at the top by `top_rung` | 4 |
| 2 - pedestal | the eight | - | 10, shot adjacent to that gain's ladder |

**Why block 2 drives the panel to level 0 rather than covering the lens.** A cover is a bench
disturbance and the attenuation is only valid while nobody moves the camera; the level is driven
from here by design. It is not perfectly dark - the backlight leaks through a black LCD (L07) -
and at 32 us what leaks is a fraction of a count at every gain in the set, printed below so it is
a measured smallness rather than an assumed zero.

Changing the grey level does not disturb the light in the way changing brightness would: the
level is the LCD's transmission and the backlight behind it is driven identically either way, so
nothing thermal moves when this loop toggles it sixteen times.

In [ ]:
leak = {g: bench["flux_counts_per_s"]["0"] * amplification(g) / amplification(SCOUT_GAIN)
             * BIAS_EXPOSURE for g in GAINS}
print("light still arriving in a 32 us block-2 frame, from the scout's level-0 flux:")
print("  " + "  ".join(f"g{g}={leak[g]:.3f}" for g in GAINS) + "  counts")
print(f"worst is {max(leak.values()):.3f} counts against a faintest rung of "
      f"{0.003 * (FULL_SCALE - pedestal(GAINS[-1])):.1f} counts\n")

t0, retaken, held = time.monotonic(), 0, 0.0
for k, g in enumerate(GAINS, 1):
    print(f"gain {g}  ({k}/{len(GAINS)})", flush=True)
    r, h = run_gain(g)
    retaken, held = retaken + r, held + h
    elapsed = time.monotonic() - t0
    print(f"  == {elapsed / 60:5.1f} min elapsed, "
          f"{elapsed / k * (len(GAINS) - k) / 60:5.1f} min to go\n", flush=True)

print(f"session: {len(list(FRAMES.glob('*.fits')))} frames, {retaken} retaken, "
      f"{held / 60:.1f} min held, {(time.monotonic() - t0) / 60:.1f} min total")

### Closing down

The cooler is switched off deliberately and the camera closed, and the panel is handed back to
whoever is holding the iPad. A hard-killed process does switch the TEC off, but leaving it that
way by accident is the difference between "warmed up on purpose" and a camera that behaves oddly
next session for reasons nobody wrote down.

Let it warm before unplugging: condensation on a cold sensor is a hardware problem, not a data
one.

**Do not move the camera or the sheets until the frame count below is right.** A short session is
recoverable while the bench is still standing and not afterwards.

In [ ]:
rig.set("CoolerOn", 0, verify=False)
rig.close()
set_level("free")                      # the panel goes back to whoever is holding it

on_disk = sorted(FRAMES.glob("*.fits"))
print("cooler off, camera closed.  Let it reach ambient before unplugging.")
print(f"{len(on_disk)} frames on disk in {FRAMES}, {planned} planned")
print(f"  flats {sum(1 for f in on_disk if f.name.startswith('flat'))}, "
      f"bias {sum(1 for f in on_disk if f.name.startswith('bias'))}")

## 3. The analysis

**Everything from here reads disk and nothing else.** No camera, no kernel state from the two
halves above: the frames, `data/session02/bench.json`, `data/session02/gate3.csv` and session 01's
`results/bias_sweep.csv` are the whole input, so the analysis can be re-run, corrected and re-run
again without ever costing a bench night. That separation is the reason the capture cells sit
above a hard boundary.

It runs `protocols/02-ptc.md`'s six analysis rules, fixed before the data existed:

| rule | what it does | published as |
|---|---|---|
| 1 | signal = plane mean minus *this gain's* block-2 pedestal, per CFA plane | `ptc_rungs.csv` |
| 2 | variance = pair-difference variance, halved | `ptc_rungs.csv` |
| 3 | `g(gain)` = the slope, per plane, with `R` **passed in** from session 01 | `ptc_gain.csv` |
| 4 | the FPN test: single-frame spatial variance against the pair variance | `ptc_constants.json` |
| 5 | the gain law: `log10 g` against gain setting | `ptc_constants.json` |
| 6 | L11's free cross-check: two-point gain on the archive, at gain 50 and 252 | `ptc_constants.json` |

Statistics on the CFA mosaic, split RGGB, never debayered; every number in ADC counts.


In [ ]:
import datetime as dt
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, spatial as SP, stats as ST

pd.set_option("display.width", 200)

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session02"
FRAMES = DATA / "frames"

RUNGS_CSV = RESULTS / "ptc_rungs.csv"        # one row per (gain, plane, rung)
GAIN_CSV = RESULTS / "ptc_gain.csv"          # one row per (gain, plane): the fitted g
NOISE_CSV = RESULTS / "read_noise_e.csv"     # session 01's R carried into electrons
CONSTANTS = RESULTS / "ptc_constants.json"   # the scalars, with provenance
SPECS = ROOT / "vendor" / "asi585specs" / "gain-curves.csv"

FULL_SCALE = ST.ADC_FULL_SCALE
GAINS = [0, 50, 100, 190, 200, 250, 300, 450]
RUNGS = [0.3, 0.5, 0.85, 1.4, 2.4, 4.0, 6.8, 11.4, 19.2, 32, 54, 90]
PLANES = SP.PLANES
OFFSET = 15
N_LADDER, N_BIAS = 4, 10

# A rung whose brightest pixels have reached the top code measures a variance
# that is too low, and pair-differencing cannot see through it because the
# clipping happens first.  The ladder was already capped for this at capture
# time from a *predicted* g; this is the same rule applied to what actually
# landed on the sensor, and it is measured rather than predicted.
CLIP_FRAC_MAX = 1e-5          # of pixels at 4095: 2.6 px of a 512x512 plane

# L25's PTC and L29's law, as predictions.  Never inputs.
L25_G = {0: 9.382, 50: 5.425, 100: 2.994, 190: 1.051, 200: 0.927, 250: 0.513, 300: 0.284}
L29_SLOPE = -0.00502           # per gain unit; the 0.1 dB law says -0.00500
L29_UNITY_GAIN = 194
L32_PRNU = 0.0061              # the fixed-pattern figure rule 4 compares against
G0_BAND = (9.38, 9.46)         # reproducing L25 / EGAIN / ZWO's GAIN=195 annotation
LAW_RESIDUAL_MAX = 0.01        # over this and g is not interpolable (protocol)

bench = json.loads((DATA / "bench.json").read_text())
gate3 = pd.read_csv(DATA / "gate3.csv").set_index("gain")
T_SAT = gate3.t_sat_s.to_dict()

# Session 01's read noise, in ADC counts at this offset.  Rule 3: R is passed
# in, never taken from the intercept (L10).
sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
sweep = sweep[sweep.offset == OFFSET]


def read_noise_counts(gain):
    """R at the nearest gain session 01 swept, in ADC counts."""
    row = sweep.iloc[int((sweep.gain - gain).abs().argmin())]
    return float(row.R_at_offset), int(row.gain)


R_COUNTS = {g: read_noise_counts(g)[0] for g in GAINS}

on_disk = sorted(FRAMES.glob("*.fits"))
print(f"{len(on_disk)} frames on disk, {len(GAINS) * (len(RUNGS) * N_LADDER + N_BIAS)} expected")
print(f"bench: grey level {bench['grey_level']}, {bench['sheets']} sheets, "
      f"brightness {bench['brightness_pct']}%, shot {bench['measured_on']}")
print("\nR passed in from session 01, ADC counts at offset "
      f"{OFFSET}: " + "  ".join(f"g{g}={R_COUNTS[g]:.3f}" for g in GAINS))


### Rules 1 and 2 - the rung table

One pass over all 464 frames, and the only pass: everything below reads the table this cell
writes rather than the pixels.

**Signal is per plane and per gain, against that gain's own block-2 pedestal** - not against
session 01's fitted pedestal, and not against the frame mean. Both shortcuts are L12 and L14
respectively, and the second one is the mistake that produced a negative dark current in a retired
attempt: a pedestal shot four hours away is a different pedestal.

**Variance is the pair difference, halved**, and there are *two disjoint pairs* per rung - frames
(0,1) and (2,3). The spread between them is the repeatability of a variance estimate, and it is
the yardstick rule 4's verdict is measured against: an FPN term only exists if the single-frame
excess is bigger than the number two honest estimates of the same variance differ by.

Three more columns come free from the same pixels: the single-frame spatial variance (rule 4), the
clipped fraction (which rungs are usable), and `g_point`, the one-rung estimate `S / (var - R^2)`
that the fit below replaces with a slope.


In [ ]:
def planes_adc(path):
    """One frame as four float64 CFA planes in ADC counts, plus its header."""
    mosaic, header = F.read(path)
    return SP.split(ST.to_adc(mosaic).astype(np.float64)), header


def pair_variance(a, b):
    """Rule 2: the temporal variance, blind to fixed pattern by construction."""
    return float(np.var(a - b, ddof=1) / 2.0)


# --- block 2: the pedestal each gain's signal is measured against ------------
pedestal, ped_scatter, R_block2 = {}, {}, {}
for g in GAINS:
    files = sorted(FRAMES.glob(f"bias_g{g:03d}_*.fits"))
    means, first = {p: [] for p in PLANES}, []
    for f in files:
        pl, _ = planes_adc(f)
        first.append(pl)
        for p in PLANES:
            means[p].append(float(pl[p].mean()))
    pedestal[g] = {p: float(np.mean(v)) for p, v in means.items()}
    ped_scatter[g] = {p: float(np.std(v, ddof=1)) for p, v in means.items()}
    # Not a re-measurement of R -- a check that this bench agrees with session 01.
    R_block2[g] = float(np.mean([np.sqrt(pair_variance(first[0][p], first[1][p]))
                                 for p in PLANES]))

print(f"{'gain':>5} {'pedestal (R G1 G2 B)':>34} {'frame-to-frame':>15} "
      f"{'R block2':>9} {'R sess01':>9}")
for g in GAINS:
    ped = "  ".join(f"{pedestal[g][p]:7.3f}" for p in PLANES)
    print(f"{g:5d} {ped:>34} {max(ped_scatter[g].values()):14.4f}  "
          f"{R_block2[g]:9.3f} {R_COUNTS[g]:9.3f}")
print("\nBlock 2 is the pedestal, not a read-noise measurement (L10): the two R columns "
      "are\na consistency check between two sessions, and the fits below use session 01's.")


In [ ]:
rows = []
for g in GAINS:
    R2 = R_COUNTS[g] ** 2
    for ri in range(len(RUNGS)):
        files = sorted(FRAMES.glob(f"flat_g{g:03d}_r{ri:02d}_*.fits"))
        if not files:
            continue
        loaded = [planes_adc(f) for f in files]
        exptime = float(np.mean([h["EXPTIME"] for _, h in loaded]))
        temp = float(np.mean([h["CCD-TEMP"] for _, h in loaded]))
        for p in PLANES:
            a = [pl[p] for pl, _ in loaded]
            var_pairs = [pair_variance(a[0], a[1]), pair_variance(a[2], a[3])]
            signal = float(np.mean([x.mean() for x in a])) - pedestal[g][p]
            rows.append({
                "gain": g, "plane": p, "rung": ri, "rung_pct_planned": RUNGS[ri],
                "pct_of_tsat": 100 * exptime / T_SAT[g], "exptime_s": exptime,
                "pedestal": pedestal[g][p], "signal": signal,
                "var_pair": float(np.mean(var_pairs)),
                "var_pair_spread": float(abs(var_pairs[0] - var_pairs[1])),
                "var_single": float(np.mean([np.var(x, ddof=1) for x in a])),
                "sat_frac": float(np.mean([(x >= FULL_SCALE).mean() for x in a])),
                "R_counts": R_COUNTS[g], "ccd_temp": temp, "n_frames": len(files),
            })
    print(f"  gain {g:3d} done", flush=True)

rungs = pd.DataFrame(rows)
rungs["g_point"] = rungs.signal / (rungs.var_pair - rungs.R_counts ** 2)
rungs["usable"] = ((rungs.sat_frac < CLIP_FRAC_MAX) & (rungs.signal > 0)
                   & (rungs.var_pair > rungs.R_counts ** 2))
# Not an exclusion -- a label.  A rung shorter than one panel refresh is the
# only kind a flickering backlight can add variance to, and the scout's
# flicker gate is what licensed shooting them.  The fit below says what
# dropping them would have changed, which is the check that gate deserves.
REFRESH_HZ = 60.0                # the panel, from the scout above
rungs["sub_refresh"] = rungs.exptime_s < 1.0 / REFRESH_HZ

rungs.to_csv(RUNGS_CSV, index=False)
print(f"\nwrote {RUNGS_CSV}  ({len(rungs)} rows, {int(rungs.usable.sum())} usable)")

dropped = rungs[~rungs.usable]
if len(dropped):
    print("\ndropped rungs (clipped, or variance below the passed-in R):")
    print(dropped.groupby(["gain", "rung"]).agg(
        planes=("plane", "count"), pct=("pct_of_tsat", "first"),
        sat=("sat_frac", "max")).to_string())
else:
    print("\nno rung dropped: the capture-time cap held at every gain.")

print("\ng_point per gain, mean over planes, by rung (e- per ADC count):")
print(rungs[rungs.usable].pivot_table(index="gain", columns="rung", values="g_point")
      .round(4).to_string())


### Rule 3 - `g(gain)`, as a slope and not as a ratio

`var = S / g + R^2`. Session 01 measured `R` at this offset and this setpoint, so the intercept is
**known**, and the fit has one free parameter: the slope. The fitted-intercept version is run
beside it and reported, because L10's synthetic test recovered **7.1 e- for a true 3.0 e-** from an
intercept while the slope from the same fit was good to 3% - the intercept is a cross-check on the
fit's health, never the read noise.

**Weighted by `1 / var^2`.** A variance estimate's error scales with the variance itself, so an
unweighted fit hands the slope to the two brightest rungs and throws away the five that were shot
specifically to constrain the low end. Weighting by the inverse square makes every rung's
*relative* error equal, which is what the ladder's geometric spacing already assumed.

The residual scatter about the fitted line is what `g_err` is built from - a spread of the points
that were actually fitted, not a formal error from an assumed noise model.


In [ ]:
def ptc_fit(signal, var, R2):
    """Rule 3.  Returns the slope-fit g with the intercept fixed at R^2, the
    free-intercept fit beside it, and the scatter of the points about the fit.

    Weights are 1/var^2: constant relative error, which is what a variance
    estimate actually has.
    """
    S, V = np.asarray(signal, float), np.asarray(var, float)
    w = 1.0 / V ** 2

    m = float(np.sum(w * S * (V - R2)) / np.sum(w * S * S))       # var - R^2 = S/g
    resid = (V - (m * S + R2)) / V
    n = len(S)
    dof = max(n - 1, 1)
    se_m = float(np.sqrt(np.sum(w * (V - m * S - R2) ** 2) / dof / np.sum(w * S * S)))

    X = np.column_stack([S, np.ones_like(S)])                    # free intercept
    sw = np.sqrt(w)
    (a, b), *_ = np.linalg.lstsq(X * sw[:, None], V * sw, rcond=None)

    return {"g": 1.0 / m, "g_err": abs(se_m / m ** 2),
            "resid_pct": float(np.sqrt(np.mean(resid ** 2)) * 100),
            "g_free": 1.0 / float(a) if a > 0 else np.nan,
            "R_fit": float(np.sqrt(b)) if b > 0 else np.nan,
            "n_rungs": int(n), "S_max": float(S.max())}


fits = []
for (g, p), d in rungs[rungs.usable].groupby(["gain", "plane"]):
    d = d.sort_values("signal")
    row = ptc_fit(d.signal, d.var_pair, R_COUNTS[g] ** 2)
    row.update({"gain": g, "plane": p, "R_counts": R_COUNTS[g],
                "R_e": row["g"] * R_COUNTS[g]})
    fits.append(row)

gain_tbl = pd.DataFrame(fits)[["gain", "plane", "g", "g_err", "resid_pct", "g_free",
                               "R_counts", "R_fit", "R_e", "n_rungs", "S_max"]]
gain_tbl = gain_tbl.sort_values(["gain", "plane"]).reset_index(drop=True)
gain_tbl.to_csv(GAIN_CSV, index=False)

per_gain = gain_tbl.groupby("gain").agg(
    g=("g", "mean"), g_sd=("g", "std"), g_err=("g_err", "mean"),
    resid_pct=("resid_pct", "mean"), R_fit=("R_fit", "mean"),
    R_counts=("R_counts", "first"), n_rungs=("n_rungs", "min"))
per_gain["plane_spread_pct"] = 100 * per_gain.g_sd / per_gain.g
per_gain["R_e"] = per_gain.g * per_gain.R_counts
per_gain["L25"] = pd.Series(L25_G)
per_gain["vs_L25"] = per_gain.g / per_gain.L25

print("g per gain, mean over the four CFA planes (e- per ADC count):")
print(per_gain[["g", "g_sd", "plane_spread_pct", "resid_pct", "R_counts", "R_fit",
                "R_e", "n_rungs", "L25", "vs_L25"]].round(4).to_string())

print("\nper plane:")
print(gain_tbl.pivot(index="gain", columns="plane", values="g").round(4).to_string())

alt = []
for (g, p), d in rungs[rungs.usable & ~rungs.sub_refresh].groupby(["gain", "plane"]):
    if len(d) >= 3:
        row = ptc_fit(d.signal, d.var_pair, R_COUNTS[g] ** 2)
        alt.append({"gain": g, "plane": p, **row})
alt = (pd.DataFrame(alt).groupby("gain")
       .agg(g_no_sub=("g", "mean"), resid_pct=("resid_pct", "mean"),
            n_rungs=("n_rungs", "min")))
alt["shift_pct"] = 100 * (alt.g_no_sub / per_gain.g - 1)
print("\nsensitivity: the same fit with every rung shorter than one 60 Hz refresh "
      "dropped.\nThe published g uses all of them; this is what the panel could have "
      "cost if it flickered:")
print(alt.round(4).to_string())

g0 = float(per_gain.g.loc[0])
print(f"\nwrote {GAIN_CSV}")
print(f"g(gain 0) = {g0:.4f} e-/ADC count, band {G0_BAND[0]}-{G0_BAND[1]}: "
      + ("REPRODUCED -- L25, header EGAIN and ZWO's GAIN=195 annotation all agree, "
         "and EGAIN becomes a standing check rather than a source"
         if G0_BAND[0] <= g0 <= G0_BAND[1] else
         "OUTSIDE the band -- the header EGAIN law is wrong for this camera and every "
         "electron figure in the repo rescales from this measurement"))
print(f"fitted intercept, as R in counts: "
      + "  ".join(f"g{int(g)}={v:.3f}" for g, v in per_gain.R_fit.items())
      + "\n  (a cross-check on the fit, never the answer -- L10)")


### Rule 5 - the gain law

Gain is in units of 0.1 dB, so `g` should fall by a factor of ten every 200 units and `log10 g`
should be a straight line of slope **-0.00500**. L29 measured **-0.00502** over gains 0-150 with
1.0% scatter, and predicted unity gain at **194** - a landmark ZWO themselves annotate as
`GAIN=195` on their published panel.

Two things are at stake. If the residual is under 1%, `g` is *interpolable*, and session 01's `R`
at all 61 of its swept gains can be carried into electrons from eight measured points. If it is
not, that economy is withdrawn and the gain set has to widen.

The law is fitted twice: with gain 450 and without it. 450 is the least constrained point in the
set - no L25 row, the vendor chart's last, and the one whose ladder was capped hardest - so if it
is out of family that must be visible rather than averaged in. L30 predicts **no step at the HCG
threshold**: the conversion-gain discontinuity lives in read noise, not in e-/ADU, so a step at
190-200 would refute L30 as well as forbid interpolation.


In [ ]:
def law_fit(gains, g_values):
    """`log10 g = intercept + slope * gain`, with the scatter it leaves behind."""
    x, y = np.asarray(gains, float), np.log10(np.asarray(g_values, float))
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    rms_dex = float(np.sqrt(np.mean(resid ** 2)))
    return {"slope": float(slope), "intercept": float(intercept),
            "rms_pct": (10 ** rms_dex - 1) * 100,
            "worst_pct": (10 ** float(np.abs(resid).max()) - 1) * 100,
            "unity_gain": float(-intercept / slope), "n": int(len(x)),
            "resid": pd.Series((10 ** resid - 1) * 100, index=np.asarray(gains))}


law = law_fit(per_gain.index, per_gain.g)
law_no450 = law_fit(per_gain.index[:-1], per_gain.g.iloc[:-1])

print(f"{'fit':>16} {'slope':>11} {'vs 0.1 dB':>10} {'vs L29':>8} "
      f"{'rms':>8} {'worst':>8} {'unity gain':>11}")
for name, f in (("all eight", law), ("without 450", law_no450)):
    print(f"{name:>16} {f['slope']:11.5f} {f['slope'] / -0.005:10.4f} "
          f"{f['slope'] / L29_SLOPE:8.4f} {f['rms_pct']:7.2f}% {f['worst_pct']:7.2f}% "
          f"{f['unity_gain']:11.1f}")

print("\nresidual per gain, % in g (all eight):")
print(law["resid"].round(2).to_string())

step = float(law["resid"].get(200, np.nan) - law["resid"].get(190, np.nan))
interpolable = law["rms_pct"] / 100 < LAW_RESIDUAL_MAX
print(f"\nHCG step, 190 -> 200: {step:+.2f}% in g against a fit scatter of "
      f"{law['rms_pct']:.2f}%")
print("  " + ("no step: L30 stands -- the conversion-gain discontinuity is in read noise "
              "and not in e-/ADU" if abs(step) < 2 * law["rms_pct"] else
              "a step at the HCG threshold: L30 is refuted and interpolation across it "
              "is forbidden"))
print(f"\nresidual {law['rms_pct']:.2f}% against the {LAW_RESIDUAL_MAX:.0%} rule: "
      + ("g is interpolable, and R(gain) in electrons is published at all of session 01's gains"
         if interpolable else
         "g is NOT interpolable -- the economy of eight gains is withdrawn and the gain "
         "set is the thing to widen"))
print(f"unity gain lands at {law['unity_gain']:.1f}; L29 predicted {L29_UNITY_GAIN}, "
      "ZWO annotate 195")

if SPECS.exists():
    specs = pd.read_csv(SPECS)
    # Gain 200 appears twice, once per conversion-gain branch.  Session 01
    # measured the threshold at 200, so 200 and up are read off the HCG row.
    specs = specs[(specs.branch == "hcg") == (specs.gain >= 200)].set_index("gain")
    vendor = per_gain.join(specs[["g_e_per_adu"]], how="left")
    vendor["vs_vendor"] = vendor.g / vendor.g_e_per_adu
    print("\nagainst ZWO's published curve (read off a plot: +/-5% at best, "
          "vendor/asi585specs/):")
    print(vendor[["g", "g_e_per_adu", "vs_vendor", "L25", "vs_L25"]].round(4).to_string())


### Rule 4 - the FPN test

This is the one measurement here that can refute a MISSION assumption rather than pin a
coefficient. Pair-differencing removes fixed pattern; a single frame's spatial variance keeps it.
So at the same signal level

```
var_single - var_pair  =  (PRNU * S)^2
```

and PRNU comes out of the *slope of the excess against `S^2`*, dimensionless, per plane. L32's
retired figure is **0.61%** over a crop.

**The verdict needs a yardstick, and the two disjoint pairs are it.** The excess only counts as a
fixed-pattern term if it is bigger than the amount two honest estimates of the same variance
differ by; below that, "single frame is noisier" is a statement about the estimator, not the
sensor. The ratio is taken over the bright rungs only, where a per-cent fixed pattern is larger
than the shot noise it has to be seen against.


In [ ]:
FPN_MIN_SIGNAL = 0.10          # of headroom; below this the excess is all estimator noise

u = rungs[rungs.usable].copy()
u["headroom_frac"] = u.signal / (FULL_SCALE - u.pedestal)
u["excess"] = u.var_single - u.var_pair
u["prnu"] = np.sqrt(u.excess.clip(lower=0)) / u.signal
u["ratio"] = u.var_single / u.var_pair
u["repeat_pct"] = 100 * u.var_pair_spread / u.var_pair

bright = u[u.headroom_frac >= FPN_MIN_SIGNAL]
prnu_by_gain = bright.groupby("gain").agg(
    prnu=("prnu", "median"), ratio=("ratio", "median"),
    excess_pct=("ratio", lambda s: 100 * (s.median() - 1)),
    repeat_pct=("repeat_pct", "median"), n=("prnu", "size"))

print(f"rungs above {FPN_MIN_SIGNAL:.0%} of headroom, per gain:")
print(prnu_by_gain.assign(prnu_pct=100 * prnu_by_gain.prnu)
      [["prnu_pct", "ratio", "excess_pct", "repeat_pct", "n"]].round(3).to_string())

prnu = float(bright.prnu.median())
prnu_sd = float(bright.prnu.std())
excess_pct = float(100 * (bright.ratio.median() - 1))
repeat_pct = float(bright.repeat_pct.median())
fpn_present = excess_pct > repeat_pct

print(f"\nPRNU {prnu:.4%} +/- {prnu_sd:.4%}, against L32's {L32_PRNU:.2%}")
print(f"single-frame variance exceeds the pair variance by {excess_pct:.2f}%; "
      f"two disjoint pairs of the same rung differ by {repeat_pct:.2f}%")
print("  " + ("AN FPN TERM EXISTS.  MISSION's first assumption -- sigma^2 is shot plus read "
              "and nothing else --\n  is refuted rather than untested, and the model gains a "
              "term that survives lengthening the sub."
              if fpn_present else
              "no fixed-pattern term above the repeatability of the estimate: sigma^2 is "
              "shot plus read\n  and nothing else, over the tested range."))

lin = u.groupby("gain").apply(
    lambda d: pd.Series({"top_pct_of_tsat": d.pct_of_tsat.max(),
                         "worst_fit_resid_pct": d.g_point.pipe(
                             lambda s: 100 * (s / s.median() - 1)).abs().max()}),
    include_groups=False)
print("\nlinearity of the pair variance in signal, as the spread of the per-rung g_point "
      "about\nits own median -- flat means variance is linear in signal to the top rung:")
print(lin.round(2).to_string())


### Rule 6 - the free cross-check (L11)

Two-point photon transfer, `g = S / (var_flat - var_bias)`, on frames that already exist: the
historic archive at gain **50** and **252**. It costs no bench time, and it is worth more than its
precision suggests - the bench and the archive have *nothing* in common. Different nights,
different optics, different light, no shared calibration, and an analysis that is two subtractions
rather than a weighted fit through twelve rungs.

The archive is a **test corpus and never a source of published constants** (`CLAUDE.md`), so this
number is recorded beside the bench value and never averaged into it. If they disagree, one of them
carries a systematic and the disagreement is the finding.

Gain 252 is not in the bench set, so its comparison goes through the fitted law - which makes it a
test of the interpolation as much as of the gain.


In [ ]:
ARCHIVE_GAINS = [50, 252]
PAIR_LEVEL_TOL = 0.02         # two frames are a "pair" if their means agree this well
CROP = 1024                   # centred, even -- the Bayer phase must not shift (L05)


def crop_centre(mosaic, n=CROP):
    h, w = mosaic.shape
    y, x = ((h - n) // 2) & ~1, ((w - n) // 2) & ~1
    return mosaic[y:y + n, x:x + n]


def archive_pairs(index, kind, gain, want=3):
    """Adjacent frames at one setting whose levels agree: an honest temporal pair."""
    d = index[(index.measured_type == kind) & (index.gain == gain)
              & (index.offset == OFFSET) & (index.xbinning == 1)
              & (index.mult16_frac == 1) & (index.status == "ok")
              & (index.sat_frac < 0.001)].sort_values(["exptime", "date_obs"])
    out = []
    for _, grp in d.groupby(["exptime", "set_temp"]):
        rows = grp.to_dict("records")
        for a, b in zip(rows[::2], rows[1::2]):
            if abs(a["mean"] - b["mean"]) / max(a["mean"], 1) < PAIR_LEVEL_TOL:
                out.append((a["path"], b["path"]))
            if len(out) >= want:
                return out
    return out


def two_point(flat_pair, bias_pair):
    """L11: g = S / (var_flat - var_bias), per CFA plane, in ADC counts."""
    fa, fb = [crop_centre(ST.to_adc(F.read(p)[0]).astype(np.float64)) for p in flat_pair]
    ba, bb = [crop_centre(ST.to_adc(F.read(p)[0]).astype(np.float64)) for p in bias_pair]
    out = {}
    for p, (f1, f2, b1, b2) in ((p, (SP.split(fa)[p], SP.split(fb)[p],
                                     SP.split(ba)[p], SP.split(bb)[p])) for p in PLANES):
        signal = float(f1.mean() + f2.mean() - b1.mean() - b2.mean()) / 2
        out[p] = signal / (pair_variance(f1, f2) - pair_variance(b1, b2))
    return out


archive = {}
try:
    index = pd.read_csv(RESULTS / "frame_index.csv")
    for g in ARCHIVE_GAINS:
        flats, biases = archive_pairs(index, "flat", g), archive_pairs(index, "bias", g, want=1)
        if not flats or not biases:
            print(f"gain {g}: no usable flat/bias pair in the index -- skipped")
            continue
        per_pair = [two_point(fp, biases[0]) for fp in flats]
        g_planes = {p: float(np.mean([r[p] for r in per_pair])) for p in PLANES}
        egain = index[(index.gain == g) & index.egain_hdr.notna()].egain_hdr.median()
        archive[g] = {"g_planes": g_planes, "g": float(np.mean(list(g_planes.values()))),
                      "n_flat_pairs": len(per_pair), "egain_hdr": float(egain)}
        print(f"gain {g}: {len(per_pair)} flat pairs, "
              + "  ".join(f"{p}={g_planes[p]:.4f}" for p in PLANES)
              + f"   mean {archive[g]['g']:.4f}   header EGAIN {egain:.4f}")
except OSError as exc:
    print(f"the archive is unreachable ({exc}); L11's cross-check is not run this pass, "
          "and\nthe bench value is published without it rather than with a substitute")

for g, a in archive.items():
    bench_g = float(per_gain.g.get(g, np.nan))
    via_law = 10 ** (law["intercept"] + law["slope"] * g)
    ref, how = (bench_g, "measured here") if np.isfinite(bench_g) else (via_law, "via the law")
    a["bench_g"] = ref
    a["agreement_pct"] = 100 * (a["g"] / ref - 1)
    print(f"\ngain {g}: archive {a['g']:.4f} vs bench {ref:.4f} ({how}) "
          f"-> {a['agreement_pct']:+.2f}%;  header EGAIN {a['egain_hdr']:.4f} "
          f"-> {100 * (a['egain_hdr'] / ref - 1):+.2f}%")
    print("  " + ("two datasets with nothing in common agree, which is stronger evidence "
                  "than either alone" if abs(a["agreement_pct"]) < 5 else
                  "they disagree: one carries a systematic.  Not averaged -- the bench value "
                  "is published\n  with the disagreement recorded beside it"))


### Publishing

Four files. `ptc_rungs.csv` and `ptc_gain.csv` are already written above - the table the fits were
made from, and the fits themselves, one row per `(gain, plane)`. This cell writes the scalars with
their provenance, and carries session 01's read noise into electrons.

**`read_noise_e.csv` is written only if the gain law passed its own residual test.** It is
session 01's `R` in ADC counts at every gain it swept inside this project's domain, multiplied by
`g` from the law - which is exactly the interpolation rule 5 licenses, and exactly what rule 5
forbids if the residual is over 1%. Nothing here re-measures read noise (L10); the electrons come
from this session's `g` and the counts come from session 01.

Gains above 450 are not carried: the domain stops there, and session 01's rows above it stay in
`bias_sweep.csv` where they were measured.


In [ ]:
first_obs = min(F.read(p)[1]["DATE-OBS"] for p in on_disk[:1] + on_disk[-1:])
measured_on = str(first_obs)[:10]
n_frames = len(on_disk)


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_frames, "measured_on": measured_on,
            "notebook": "05_ptc.ipynb", "note": note}


g_per_gain = {int(g): round(float(v), 5) for g, v in per_gain.g.items()}
g_err_per_gain = {int(g): round(float(v), 5) for g, v in per_gain.g_err.items()}

constants = {
    "system_gain": constant(
        g_per_gain, "e- per ADC count", g_err_per_gain,
        f"rule 3 of protocols/02-ptc.md: the slope of pair-difference variance against "
        f"signal, weighted 1/var^2, with the intercept fixed at session 01's R^2 (L10).  "
        f"Mean over the four CFA planes; per-plane values are in ptc_gain.csv and spread "
        f"{per_gain.plane_spread_pct.max():.2f}% at worst.  Uncertainty is the scatter of "
        f"the fitted rungs, not a formal error"),
    "g_at_gain0": constant(
        round(g0, 4), "e- per ADC count", round(float(per_gain.g_err.loc[0]), 4),
        f"the anchor of the gain law.  L25 predicted 9.382 and the reproduction band was "
        f"{G0_BAND[0]}-{G0_BAND[1]}; measured {g0:.4f}, ratio to L25 "
        f"{g0 / L25_G[0]:.4f}.  "
        + ("Reproduced, so header EGAIN is a standing check rather than a source"
           if G0_BAND[0] <= g0 <= G0_BAND[1] else
           "Outside the band: every electron figure in the repo rescales from this")),
    "gain_law": constant(
        {"slope": round(law["slope"], 6), "intercept_log10_g": round(law["intercept"], 5),
         "unity_gain": round(law["unity_gain"], 1),
         "residual_pct": round(law["rms_pct"], 3),
         "slope_without_gain450": round(law_no450["slope"], 6)},
        "log10(e- per ADC count) per gain unit", round(law["worst_pct"] / 100, 5),
        f"rule 5: g = 10 ** (intercept + slope * gain), fitted over "
        f"{law['n']} gains.  The 0.1 dB law predicts -0.00500 and L29 measured -0.00502; "
        f"this is {law['slope']:.5f}, a ratio of {law['slope'] / -0.005:.4f}.  Residual "
        f"{law['rms_pct']:.2f}% against the 1% rule, so g is "
        + ("interpolable across the domain; uncertainty is the worst residual as a fraction"
           if interpolable else
           "NOT interpolable -- the gain set must widen before any number is read off "
           "between measured points")),
    "hcg_step_in_gain": constant(
        round(step, 3), "% change in g across the HCG threshold", round(law["rms_pct"], 3),
        f"rule 5's refutation test for L30: the conversion-gain discontinuity should appear "
        f"in read noise and not in e-/ADU.  Residual at gain 200 minus residual at 190, "
        f"against a fit scatter of {law['rms_pct']:.2f}%.  "
        + ("No step: L30 stands" if abs(step) < 2 * law["rms_pct"] else
           "A step: L30 is refuted and interpolation across the threshold is forbidden")),
    "prnu": constant(
        round(prnu, 5), "fraction of signal (fixed-pattern sigma / signal)", round(prnu_sd, 5),
        f"rule 4: sqrt(single-frame spatial variance - pair-difference variance) / signal, "
        f"median over every plane and gain above {FPN_MIN_SIGNAL:.0%} of headroom.  L32 "
        f"predicted 0.61%.  The single-frame variance runs {excess_pct:.2f}% above the pair "
        f"variance where two disjoint pairs of the same rung differ by {repeat_pct:.2f}%, so "
        + ("an FPN term exists and MISSION's first assumption is refuted"
           if fpn_present else
           "the excess is inside the repeatability of the estimate and MISSION's first "
           "assumption holds over the tested range")),
    "fpn_term_present": constant(
        bool(fpn_present), "boolean", None,
        "whether sigma^2 is shot plus read and nothing else, over the tested range.  The "
        "yardstick is the pair repeatability, not zero: two honest estimates of the same "
        "variance differ, and an excess below that difference is a statement about the "
        "estimator rather than about the sensor"),
    "archive_cross_check": constant(
        {str(g): {"g": round(a["g"], 4), "bench_g": round(a["bench_g"], 4),
                  "agreement_pct": round(a["agreement_pct"], 2),
                  "egain_hdr": round(a["egain_hdr"], 4),
                  "flat_pairs": a["n_flat_pairs"]} for g, a in archive.items()} or None,
        "e- per ADC count", None,
        "rule 6 / L11: two-point photon transfer g = S / (var_flat - var_bias) on the "
        "historic archive at gain 50 and 252, cropped to a central 1024x1024.  A cross-check "
        "on frames with nothing in common with the bench -- recorded beside the bench value "
        "and never averaged into it, because no published constant comes from the archive.  "
        "Gain 252 is not in the bench set, so its comparison runs through the fitted law and "
        "tests the interpolation too"),
    "vendor_prediction": constant(
        {str(int(g)): L25_G[g] for g in L25_G}, "e- per ADC count", None,
        "L25's retired 61-point sweep, kept as the prediction this session either reproduced "
        "or refuted -- not an input.  Measured/predicted per gain: "
        + ", ".join(f"{int(g)}:{v:.3f}" for g, v in per_gain.vs_L25.dropna().items())
        + ".  ZWO's own curve is in vendor/asi585specs/ and is read off a plot, +/-5% at best"),
}

with open(CONSTANTS, "w") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS} with {len(constants)} constants, provenance on each")

# --- R(gain) in electrons, if and only if the law earned it -----------------
if interpolable:
    domain = sweep[(sweep.gain <= 450)].sort_values("gain")
    noise = pd.DataFrame({
        "gain": domain.gain.astype(int),
        "R_counts": domain.R_at_offset.astype(float),
        "g_e_per_adu": 10 ** (law["intercept"] + law["slope"] * domain.gain),
    })
    noise["R_e"] = noise.R_counts * noise.g_e_per_adu
    noise["measured_g"] = noise.gain.map(g_per_gain)
    noise.to_csv(NOISE_CSV, index=False)
    print(f"wrote {NOISE_CSV}: session 01's R at {len(noise)} gains, carried into electrons "
          f"by the law")
    print(noise[noise.gain.isin(GAINS)].round(4).to_string(index=False))
else:
    if NOISE_CSV.exists():
        NOISE_CSV.unlink()
    print(f"{NOISE_CSV.name} NOT written: the law's residual is {law['rms_pct']:.2f}%, over "
          f"the {LAW_RESIDUAL_MAX:.0%} rule.\nR in electrons is available only at the eight "
          "measured gains, in ptc_gain.csv, until the gain set widens")

print(f"\nsession 02 published: {RUNGS_CSV.name}, {GAIN_CSV.name}, {CONSTANTS.name}"
      + (f", {NOISE_CSV.name}" if interpolable else ""))
print("Next: 06_ptc_read explains what these files say.  It measures nothing, and where it "
      "disagrees\nwith results/, results/ is right.")
